# 02 - HTTP Methods and Idempotency

## 1. HTTP Methods

**Definition:** HTTP methods tell the server **what action** to perform on a resource.

| Method | Action | Example |
|---|---|---|
| GET | Read data | GET /orders/5 |
| POST | Create new | POST /orders |
| PUT | Replace whole resource | PUT /orders/5 |
| PATCH | Update some fields | PATCH /orders/5 |
| DELETE | Remove | DELETE /orders/5 |

**PUT vs PATCH:**
- **PUT** replaces the whole resource. Fields you leave out may be wiped.
- **PATCH** changes only the fields you send.
- *Example:* to change only the status of an order, use PATCH with {"status": "shipped"}.

**Real-world example:** On a shopping site, browsing products is GET, placing an order is POST, changing the delivery address is PATCH, and cancelling the order is DELETE.

**Interview line:** "GET reads, POST creates, PUT replaces the whole resource, PATCH updates part of it, and DELETE removes it."

In [ ]:
# ==================================================
# 1. PUT vs PATCH - what happens to the data
# ==================================================
order = {"id": 5, "item": "Laptop", "address": "Hyderabad", "status": "paid"}
print("[ORIGINAL] ", order)

# PATCH: only the sent field changes
patched = {**order, "status": "shipped"}
print("[PATCH]    ", patched)

# PUT: the whole resource is replaced by what you send
put_body = {"id": 5, "item": "Laptop", "status": "shipped"}
replaced = put_body
print("[PUT]      ", replaced)
print("[NOTE]      address is gone after PUT because it was not sent")

## 2. Idempotency

**Definition:** An operation is **idempotent** if doing it once or many times leaves the server in the same state.

| Method | Idempotent? | Why |
|---|---|---|
| GET | Yes | Only reads, nothing changes |
| PUT | Yes | Replacing with the same data gives the same result |
| DELETE | Yes | Once deleted, it stays deleted |
| PATCH | Usually | Setting status = shipped twice is the same; incrementing a counter is not |
| POST | No | Each call creates a new record |

**The problem with POST:** If the network drops and the client retries, the user can be charged twice.

**Fix: idempotency key**
- The client sends a unique key in a header, e.g. Idempotency-Key: abc123
- The server stores the key with the result
- If the same key comes again, the server returns the saved result instead of processing again

**Real-world example:** Razorpay and Stripe accept an idempotency key on payment requests, so a retried "Pay" click never charges twice.

**Interview line:** "GET, PUT and DELETE are idempotent; POST is not. To make POST safe for retries, I use an idempotency key so duplicate requests return the saved result."

In [ ]:
# ==================================================
# 2. IDEMPOTENCY KEY - stopping a double charge
# ==================================================
processed = {}   # stores idempotency_key -> saved result
payments = []    # actual charges made

def create_payment(idempotency_key, amount):
    if idempotency_key in processed:
        print(f"[DUPLICATE] key={idempotency_key} -> returning saved result")
        return processed[idempotency_key]
    payment = {"payment_id": len(payments) + 1, "amount": amount, "status": "success"}
    payments.append(payment)
    processed[idempotency_key] = payment
    print(f"[NEW]       key={idempotency_key} -> charged {amount}")
    return payment

print("=" * 55)
create_payment("abc123", 500)   # first click
create_payment("abc123", 500)   # retry after network drop
create_payment("xyz789", 300)   # a genuinely new payment
print("=" * 55)
print(f"[RESULT] total charges made: {len(payments)}")

## Key Takeaways

- GET reads, POST creates, PUT replaces, PATCH updates part, DELETE removes
- PUT sends the **whole** resource; PATCH sends **only changed fields**
- Idempotent = same result whether called once or many times
- GET, PUT, DELETE are idempotent; POST is not
- Make POST retry-safe with an **Idempotency-Key** header